In [1]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random
import tensorflow as tf

In [2]:
# Get the project directory
PROJECT_DIR = Path.cwd().parents[0]

In [3]:
# Sizes
IMG_SIZE = 64
BATCH_SIZE = 32

In [4]:
DIGITS = list(range(1, 10))

FONT_PATHS = [
    "/System/Library/Templates/Data/Library/Fonts/Arial Unicode.ttf",
    "/System/Library/Fonts/Helvetica.ttc",
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
]

In [5]:
def generate_digit_image(digit):
    img = Image.new("L", (IMG_SIZE, IMG_SIZE), 255)
    draw = ImageDraw.Draw(img)

    font_path = random.choice(FONT_PATHS)
    font_size = random.randint(30, 60)

    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()

    text = str(digit)

    # center text
    bbox = draw.textbbox((0, 0), text, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]

    x = (IMG_SIZE - w) // 2 + random.randint(-5, 5)
    y = (IMG_SIZE - h) // 2 + random.randint(-5, 5)

    draw.text((x, y), text, fill=0, font=font)

    # blur
    if random.random() < 0.4:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0, 1.5)))

    # convert to array
    arr = np.array(img).astype(np.float32)

    # noise
    if random.random() < 0.5:
        arr += np.random.normal(0, 20, arr.shape)

    arr = np.clip(arr, 0, 255)

    # convert to RGB (MobileNetV2 expects 3 channels)
    arr = np.stack([arr, arr, arr], axis=-1)

    return arr.astype(np.float32), digit - 1

In [6]:
def data_generator(mode: str = "train"):
    while True:
        digit = random.choice(DIGITS)
        x, y = generate_digit_image(digit)
        yield x, y

In [7]:
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

dataset = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

2026-06-06 14:19:01.376018: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-06 14:19:01.376042: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-06-06 14:19:01.376045: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1780773541.376064  677043 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1780773541.376089  677043 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
val_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
# Customize a model from the base MobileNetV2 model
NUM_CLASSES = 9

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.RandomRotation(0.05)(inputs)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomContrast(0.2)(x)

x = keras.applications.mobilenet_v2.preprocess_input(x)

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

/var/folders/vb/gdmzt8xs5bb2rp7k5ht40ync0000gn/T/ipykernel_14743/3551942551.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


In [10]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary of the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 64, 64, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 64, 64, 3)      │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,553 (8.93 MB)

 Trainable params: 82,569 (322.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [12]:
history = model.fit(
    dataset,
    steps_per_epoch=200,
    epochs=20,
    callbacks=[callback]
)

Epoch 1/20


2026-06-06 14:19:02.657665: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.3239 - loss: 2.1956
Epoch 2/20
  7/200 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.5643 - loss: 1.2413

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.5981 - loss: 1.1371
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.7100 - loss: 0.7937
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.7770 - loss: 0.6194
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8231 - loss: 0.5047
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8473 - loss: 0.4326
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8636 - loss: 0.3860
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8820 - loss: 0.3465
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8936 - loss: 0.3190
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.8936 - loss: 0.3010
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9036 - loss: 0.2749
Epoch 12/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9117 - loss: 0.2576
Epoch 13/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/st

In [14]:
# Evaluate the model
test_loss, test_acc = model.evaluate(val_ds, steps=5, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

5/5 - 0s - 25ms/step - accuracy: 0.9500 - loss: 0.1560

Test Accuracy: 95.00%


In [15]:
# Save the model and its parameters
model.save(f'{PROJECT_DIR}/models/font_recognition_MobileNetV2.keras')